# Clasificación de Noticias

## Data Preparation

### TF-IDF

En esta sección preparamos los datos para entrenar modelos de clasificación. Cargamos las representaciones TF-IDF generadas previamente que capturan la importancia de cada término en los documentos. Estos vectores servirán como features (X) para predecir los topics de las noticias financieras (y). Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y test (10%) para evaluar el rendimiento de los modelos.

In [63]:
import pandas as pd
import os

tfidf_data = pd.read_parquet('../../Representacion_del_lenguaje/processed_data/tfidf_embeddings.parquet')
tfidf_cols = [c for c in tfidf_data.columns if c.startswith("tfidf_")]
X = tfidf_data[tfidf_cols].values
print(f"Dimensiones de X: {X.shape}")

Dimensiones de X: (5160, 5000)


In [64]:
import numpy as np

df = pd.read_csv("../../../data/definitivos/INDEX_ALL_scrapped_filtrado.csv")
y = df["topic"].values

print("Total de textos:", y.size)
print("\nDistribución de textos por topic:")
print(df["topic"].value_counts().sort_index())

Total de textos: 5160

Distribución de textos por topic:
topic
Business Growth and Cloud Infrastructure in the AI Industry    1539
Financial and Market News and Corporate Sales                  2257
Informal / Conversational Lenguaje                              152
Quantum Computing and Military Technology                       101
Stock Market and Trading                                       1111
Name: count, dtype: int64


In [65]:
from sklearn.model_selection import train_test_split

X_train_tfidf, X_temp, y_train_tfidf, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val_tfidf, X_test_tfidf, y_val_tfidf, y_test_tfidf = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Textos (variable predictoria):\n", "Training", X_train_tfidf.shape, "Validation", X_val_tfidf.shape, "Test", X_test_tfidf.shape)
print("Topics (variable a predecir):\n", "Training", y_train_tfidf.size, "Validation", y_val_tfidf.size, "Test", y_test_tfidf.size)

Textos (variable predictoria):
 Training (4128, 5000) Validation (516, 5000) Test (516, 5000)
Topics (variable a predecir):
 Training 4128 Validation 516 Test 516


### Emb. No Contextuales

In [53]:
from gensim.models import KeyedVectors
import os

w2v_model = KeyedVectors.load('../../Representacion_del_lenguaje/embeddings/NoContext/w2v_sg.kv', mmap='r')
print(w2v_model)

KeyedVectors<vector_size=300, 16863 keys>


En esta sección trabajamos con embeddings de Word2Vec. A diferencia de TF-IDF que captura la importancia estadística de los términos, Word2Vec aprende representaciones densas de 300 dimensiones que codifican información semántica: palabras con significados similares tienen vectores cercanos en el espacio vectorial. Cargamos el modelo Word2Vec entrenado previamente con Skip-Gram sobre nuestro corpus de noticias financieras, que contiene 16,863 palabras del vocabulario.

In [54]:
train_set = pd.read_csv('../../../data/definitivos/splits/train.csv')
test_set = pd.read_csv('../../../data/definitivos/splits/test.csv')

print("Tamaño del conjunto de entrenamiento:", train_set.shape)
print("Tamaño del conjunto de prueba:", test_set.shape)

Tamaño del conjunto de entrenamiento: (4128, 7)
Tamaño del conjunto de prueba: (1032, 7)


In [60]:
import pandas as pd

processed_texts = pd.read_parquet('../../Representacion_del_lenguaje/processed_data/preprocNoContext_embeddings.parquet').drop_duplicates(subset=['article_text'], keep='first')

train_combined = train_set.merge(
    processed_texts[['article_text', 'ncEmbedText']], 
    on='article_text', 
    how='left'
)

test_combined = test_set.merge(
    processed_texts[['article_text', 'ncEmbedText']], 
    on='article_text', 
    how='left'
)

X_train = train_combined['ncEmbedText']
y_train = train_combined['topic']

X_test = test_combined['ncEmbedText']
y_test = test_combined['topic']

np.save('X_w2v_train.npy', X_train.to_numpy())
np.save('X_w2v_test.npy', X_test.to_numpy())
np.save('y_w2v_train.npy', y_train.to_numpy())
np.save('y_w2v_test.npy', y_test.to_numpy())

print(f'Train processed texts:\n {X_train.head(5)}')
print(X_train.size)
print(f'\nTest processed texts:\n {X_test.head(5)}')
print(X_test.size)

Train processed texts:
 0    reuters has partnered with broadcom produce it...
1    ceo transition take effect march year slingerl...
2    targets apple google and microsoft over online...
3    mcdonald corporation nyse mcd included among t...
4    kelley blue book executive editor brian moody ...
Name: ncEmbedText, dtype: object
4128

Test processed texts:
 0    key takeaways leader saw its stock drop tuesda...
1    apple aapl shares have gained percent over the...
2    steven scheer jerusalem reuters high tech comp...
3    elon musk back atop billionaires list tesla ce...
4    stocks pull back from recent record highs simp...
Name: ncEmbedText, dtype: object
1032


In [56]:
import nltk

def texto_a_vector(text, model):
    # Tokenizar
    words = nltk.word_tokenize(text.lower()) 
    
    # Vectores de las palabras
    word_vectors = [model[w] for w in words if w in model]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(word_vectors, axis=0)

Para convertir los textos preprocesados en vectores, definimos una función que: tokeniza cada texto, obtiene los vectores Word2Vec de cada palabra presente en el vocabulario del modelo, y promedia estos vectores para obtener una representación única del documento de 300 dimensiones. Si ninguna palabra del texto está en el vocabulario, devolvemos un vector de ceros. Este método de averaging es simple pero efectivo, capturando la semántica general del documento.

In [57]:
Xw2v_train = np.array([texto_a_vector(text, w2v_model) for text in X_train])
yw2v_train = y_train

X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

Xw2v_val = np.array([texto_a_vector(text, w2v_model) for text in X_val])
Xw2v_test = np.array([texto_a_vector(text, w2v_model) for text in X_test])
yw2v_val = y_val
yw2v_test = y_test

print(f"Dimensiones de X_w2v para training: {Xw2v_train.shape}")

Dimensiones de X_w2v para training: (4128, 300)


In [58]:
print(f"Dimensiones de X_w2v para validation: {Xw2v_val.shape}")
print(f"Dimensiones de X_w2v para test: {Xw2v_test.shape}")

Dimensiones de X_w2v para validation: (516, 300)
Dimensiones de X_w2v para test: (516, 300)


Cada documento queda representado por un vector de 300 dimensiones (mucho más denso que los 5000 de TF-IDF), que captura el significado semántico promedio de sus palabras. Luego dividimos los datos en los mismos conjuntos de entrenamiento (80%), validación (10%) y test (10%) que usamos con TF-IDF, para asegurar comparabilidad entre experimentos.

Guardamos los splits generados para utilizar los mismos en las aplicaciones Deep Learning, para una misma medición justa.

## *Multiclass Logistic Regression*

Vamos primero a implementar una Regresión Logística Multinomial, un modelo clásico de machine learning que es eficiente y efectivo para clasificación de texto. Utilizamos el solver 'lbfgs' que es apropiado para problemas multiclase. Este modelo baseline nos permitirá establecer un punto de referencia para comparar con modelos más complejos posteriormente.

In [66]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(multi_class='multinomial',
                             solver='lbfgs',
                             max_iter=1000,
                             random_state=42)

tfidf_log_reg = log_reg.fit(X_train_tfidf, y_train_tfidf)

c:\Users\Iñigo Peña\Desktop\Clase\2025-26\ProcesamientoDelLenguajeNatural\FinTracker\ftenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [67]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = tfidf_log_reg.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test_tfidf, y_pred))
print("Macro-F1:", f1_score(y_test_tfidf, y_pred, average="macro"))

Accuracy: 0.9050387596899225
Macro-F1: 0.8245296524292908


 El **Accuracy** indica el porcentaje total de predicciones correctas, mientras que el **Macro-F1** promedia el F1-score de todas las clases sin ponderar por frecuencia, siendo especialmente útil cuando hay desbalance entre topics. El valor Macro-F1 no es muy lejano al Accuracy, por lo que el modelo no muestra dificultades con clases minoritarias, como podrian ser *"Informal / Conversational Lenguaje"* o *"Quantum Computing and Military Technology"* en nuestro caso.

In [68]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = log_reg.predict(X_val_tfidf)

acc = accuracy_score(y_val_tfidf, y_val_pred)
f1_macro = f1_score(y_val_tfidf, y_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(y_val_tfidf, y_val_pred))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(y_val_tfidf, y_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.92      0.91      0.91       159
              Financial and Market News and Corporate Sales       0.89      0.96      0.92       226
                         Informal / Conversational Lenguaje       1.00      0.50      0.67        18
                  Quantum Computing and Military Technology       1.00      0.67      0.80        12
                                   Stock Market and Trading       0.92      0.88      0.90       101

                                                   accuracy                           0.91       516
                                                  macro avg       0.94      0.78      0.84       516
                                               weighted avg       0.91      0.91      0.90       516


Confusion Matrix (Validation):
[[144  12   0   0  

### Utilizando Random Subsampling (TF-IDF)

Añadimos una sola prueba de subsampling aleatorio del conjunto de entrenamiento TF-IDF para balancear clases y comprobar si realmente ayuda a las minoritarias.

In [70]:
from collections import Counter

def random_subsample(X_data, y_data, random_state=42):
    rng = np.random.default_rng(random_state)
    y_array = np.array(y_data)
    classes, counts = np.unique(y_array, return_counts=True)
    min_count = counts.min()
    sampled_idx = np.concatenate([
        rng.choice(np.where(y_array == cls)[0], size=min_count, replace=False)
        for cls in classes
    ])
    rng.shuffle(sampled_idx)
    return X_data[sampled_idx], y_array[sampled_idx]

In [71]:
X_train_sub, y_train_sub = random_subsample(X_train_tfidf, y_train_tfidf, random_state=42)
print("Distribución tras subsampling (training TF-IDF):", Counter(y_train_sub))
print("Tamaño training balanceado:", X_train_sub.shape, y_train_sub.size)

Distribución tras subsampling (training TF-IDF): Counter({'Quantum Computing and Military Technology': 74, 'Financial and Market News and Corporate Sales': 74, 'Business Growth and Cloud Infrastructure in the AI Industry': 74, 'Informal / Conversational Lenguaje': 74, 'Stock Market and Trading': 74})
Tamaño training balanceado: (370, 5000) 370


In [75]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

log_reg_sub = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
tfidf_log_reg_sub = log_reg_sub.fit(X_train_sub, y_train_sub)

y_val_sub_pred = tfidf_log_reg_sub.predict(X_val_tfidf)
y_test_sub_pred = tfidf_log_reg_sub.predict(X_test_tfidf)

print("Validation - Accuracy:", accuracy_score(y_val_tfidf, y_val_sub_pred))
print("Validation - Macro-F1:", f1_score(y_val_tfidf, y_val_sub_pred, average="macro"))
print("Test - Accuracy:", accuracy_score(y_test_tfidf, y_test_sub_pred))
print("Test - Macro-F1:", f1_score(y_test_tfidf, y_test_sub_pred, average="macro"))

print("Classification Report (Validation con subsampling):")
print(classification_report(y_val_tfidf, y_val_sub_pred))

Validation - Accuracy: 0.8158914728682171
Validation - Macro-F1: 0.7675798225519112
Test - Accuracy: 0.8081395348837209
Test - Macro-F1: 0.7610666596660555
Classification Report (Validation con subsampling):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.83      0.82      0.83       159
              Financial and Market News and Corporate Sales       0.87      0.80      0.84       226
                         Informal / Conversational Lenguaje       0.64      0.89      0.74        18
                  Quantum Computing and Military Technology       0.48      0.92      0.63        12
                                   Stock Market and Trading       0.79      0.81      0.80       101

                                                   accuracy                           0.82       516
                                                  macro avg       0.72      0.85   

El subsampling recorta datos de las clases mayoritarias y, como se observa en las metricas anteriores, no mejora (o incluso empeora) las clases minoritarias. Al perder ejemplos, el modelo generaliza peor y no compensa el desbalance; por eso no resulta útil en este caso.

### Usando Word2Vec

In [76]:
w2v_log_reg = log_reg.fit(Xw2v_train, yw2v_train)

c:\Users\Iñigo Peña\Desktop\Clase\2025-26\ProcesamientoDelLenguajeNatural\FinTracker\ftenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [77]:
yw2v_pred = tfidf_log_reg.predict(Xw2v_test)
print("Accuracy:", accuracy_score(yw2v_test, yw2v_pred))
print("Macro-F1:", f1_score(yw2v_test, yw2v_pred, average="macro"))

Accuracy: 0.8178294573643411
Macro-F1: 0.7756491207623138


Los resultados con Word2Vec muestran métricas ligeramente diferentes respecto a TF-IDF. Mientras TF-IDF representa cada documento con un vector disperso de 5000 dimensiones basado en frecuencias de términos, Word2Vec usa vectores densos de solo 300 dimensiones que capturan relaciones semánticas entre palabras. La diferencia en rendimiento dependerá de si las relaciones semánticas capturadas por Word2Vec son más informativas que las estadísticas de frecuencia de TF-IDF para distinguir entre los diferentes topics financieros.

In [78]:
yw2v_val_pred = log_reg.predict(Xw2v_val)

acc = accuracy_score(yw2v_val, yw2v_val_pred)
f1_macro = f1_score(yw2v_val, yw2v_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(yw2v_val, yw2v_val_pred))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(yw2v_val, yw2v_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.84      0.85      0.84       155
              Financial and Market News and Corporate Sales       0.85      0.89      0.87       230
                         Informal / Conversational Lenguaje       0.89      0.57      0.70        14
                  Quantum Computing and Military Technology       1.00      0.43      0.60         7
                                   Stock Market and Trading       0.85      0.83      0.84       110

                                                   accuracy                           0.85       516
                                                  macro avg       0.89      0.71      0.77       516
                                               weighted avg       0.85      0.85      0.84       516


Confusion Matrix (Validation):
[[131  16   1   0  

En los resultados de Word2Vec, observamos diferencias significativas en el rendimiento con TF-IDF: mientras que ambos métodos logran buenos resultados en las clases mayoritarias, Word2Vec puede mostrar un comportamiento distinto en las clases minoritarias debido a su capacidad de capturar similitudes semánticas. TF-IDF presenta mejores tresultados, lo cual indica que los términos específicos y su peso discriminativo son más importantes que la información semántica contextual. La Confusion Matrix revela qué clases se benefician más de cada tipo de representación, ayudándonos a entender cuál método captura mejor las características distintivas de cada topic financiero.

## Linear SVM

Implementamos ahora una Support Vector Machine (SVM) lineal, un clasificador robusto que busca el hiperplano óptimo que maximiza el margen entre clases. En espacios de alta dimensionalidad como el generado por TF-IDF (5000 features), los datos suelen ser linealmente separables, lo que puede hacer que SVM lineal sea particularmente efectivo. El parámetro C controla el trade-off entre maximizar el margen y minimizar errores de clasificación; valores más altos penalizan más los errores.

In [81]:
from sklearn.svm import LinearSVC

svm = LinearSVC(C=1.0)
tfidf_svm = svm.fit(X_train_tfidf, y_train_tfidf)

In [82]:
from sklearn.metrics import accuracy_score, f1_score

pred = tfidf_svm.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test_tfidf, pred))
print("Macro-F1:", f1_score(y_test_tfidf, pred, average="macro"))

Accuracy: 0.9282945736434108
Macro-F1: 0.8715570679691778


Los resultados de SVM muestran una mejora respecto a la Regresión Logística. Esto se explica por la naturaleza del problema: con la alta dimensionalidad de TF-IDF, las 5 clases se vuelven linealmente separables en este espacio de alta dimensión. SVM es especialmente bueno aprovechando esta separabilidad lineal al encontrar el hiperplano óptimo que maximiza el margen entre clases, lo que resulta en mejor generalización.

In [83]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = svm.predict(X_val_tfidf)

acc = accuracy_score(y_val_tfidf, y_val_pred)
f1_macro = f1_score(y_val_tfidf, y_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(y_val_tfidf, y_val_pred))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(y_val_tfidf, y_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.93      0.93      0.93       159
              Financial and Market News and Corporate Sales       0.92      0.94      0.93       226
                         Informal / Conversational Lenguaje       0.88      0.83      0.86        18
                  Quantum Computing and Military Technology       1.00      0.75      0.86        12
                                   Stock Market and Trading       0.93      0.90      0.91       101

                                                   accuracy                           0.92       516
                                                  macro avg       0.93      0.87      0.90       516
                                               weighted avg       0.92      0.92      0.92       516


Confusion Matrix (Validation):
[[148   8   0   0  

### Usando Word2Vec

In [84]:
w2v_svm = svm.fit(Xw2v_train, yw2v_train)

pred = w2v_svm.predict(Xw2v_test)
print("Accuracy:", accuracy_score(yw2v_test, pred))
print("Macro-F1:", f1_score(yw2v_test, pred, average="macro"))

Accuracy: 0.8488372093023255
Macro-F1: 0.8136794036835466


Repetimos el proceso de aplicar SVM lineal a los vectores Word2Vec de 300 dimensiones. A diferencia de TF-IDF, con Word2Vec trabajamos en un espacio más compacto pero mas valioso semánticamente. SVM aprovecha las relaciones semánticas capturadas por Word2Vec para encontrar el hyperplane que mejor representa los datos.

In [85]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_pred = svm.predict(Xw2v_val)

acc = accuracy_score(yw2v_val, y_val_pred)
f1_macro = f1_score(yw2v_val, y_val_pred, average="macro")

print("\nClassification Report (Validation):")
print(classification_report(yw2v_val, y_val_pred))
print("\nConfusion Matrix (Validation):")
print(confusion_matrix(yw2v_val, y_val_pred))


Classification Report (Validation):
                                                             precision    recall  f1-score   support

Business Growth and Cloud Infrastructure in the AI Industry       0.85      0.86      0.86       155
              Financial and Market News and Corporate Sales       0.88      0.92      0.90       230
                         Informal / Conversational Lenguaje       1.00      0.64      0.78        14
                  Quantum Computing and Military Technology       1.00      0.71      0.83         7
                                   Stock Market and Trading       0.92      0.88      0.90       110

                                                   accuracy                           0.88       516
                                                  macro avg       0.93      0.80      0.85       516
                                               weighted avg       0.88      0.88      0.88       516


Confusion Matrix (Validation):
[[133  18   0   0  

En el Classification Report podemos ver que el rendimiento en las clases minoritarias muestra una diferencia: TF-IDF mantiene su ventaja, confirmando que los términos discriminativos específicos son más informativos que relaciones semánticas para este problema.